# Notebook 01 — EDA & Advanced Feature Engineering
### Airbnb Nightly Price Prediction (Regression)

**Module:** ITI113 Machine Learning & Operations
**Focus Area:** A — Data & EDA
**Pipeline stages covered:** 1. Data Storage → 2. Preprocessing
**Estimated runtime:** 8–12 minutes on `ml.t3.medium`

---

## Problem statement

Predict the **nightly listing price** of an Airbnb property from its structural, geographic,
host and review attributes. The target is **continuous**, so this is a **regression** problem —
every design decision below (metrics, transforms, outlier policy, error analysis) is chosen for
a continuous target, not a class label.

## What this notebook does

1. Loads `Listings.csv` (279,712 rows × 33 columns) from S3 and version-pins the raw snapshot.
2. Runs a formal **data suitability assessment** against the regression objective.
3. Performs deep EDA aimed specifically at *price* — distribution shape, spatial structure,
   confounding, and pricing anomalies.
4. Cleans the data: currency normalisation, outlier policy, missing-value strategy,
   sentinel-value repair.
5. Engineers an advanced feature set, with an explicit written justification (logical **and**
   mathematical) for every family of features.
6. Splits stratified by city and writes the processed matrices to S3 for Notebook 02.

## Prerequisites

- SageMaker Studio, kernel **Python 3 (Data Science 3.0)**, instance `ml.t3.medium`
- Execution role has S3 read/write on the team prefix
- `Listings.csv` uploaded to `s3://<bucket>/<prefix>/raw/Listings.csv`

## How to adapt this for your own project

- Change `TEAM_ID`, `STUDENT_ID`, `PROJECT_NAME` in the configuration cell only.
- The `FX_RATES` dictionary is a **modelling assumption** and is version-tagged — see §4.2.
- Nothing else in the notebook needs to change.

In [4]:
# Run once, then restart the kernel before continuing.
%pip install -q -U pandas numpy scikit-learn matplotlib seaborn boto3 sagemaker

Note: you may need to restart the kernel to use updated packages.


In [2]:
# After running this cell, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore

  Using cached sagemaker-2.257.6-py3-none-any.whl.metadata (19 kB)


  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)


  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)


  Using cached sagemaker_core-1.0.78-py3-none-any.whl.metadata (4.9 kB)


Using cached sagemaker-2.257.6-py3-none-any.whl (1.7 MB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached sagemaker_core-1.0.78-py3-none-any.whl (444 kB)


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.7.0 which is incompatible.
autogluon-multimodal 1.5.0 requires jsonschema<4.24,>=4.18, but you have jsonschema 4.26.0 which is incompatible.
autogluon-multimodal 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.5.2 which is incompatible.
autogluon-multimodal 1.5.0 requires Pillow<12,>=10.0.1, but you have pillow 12.3.0 which is incom

Note: you may need to restart the kernel to use updated packages.


## 0. Configuration

Team / student identifiers follow the same convention as the MLflow App setup notebook
(`01A_setup_sagemaker_mlflow_app`) so that S3 prefixes, MLflow experiments and IAM tags all line up.

In [1]:
import io
import json
import warnings
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import sagemaker

warnings.filterwarnings("ignore")

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

# ============================================================
# Team / student settings — change these only
# ============================================================
TEAM_ID = "team14"
STUDENT_ID = "s1402"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "airbnb-pricing"

BUCKET = "nyp-26s1-iti113"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

# ============================================================
# Modelling contract
# ============================================================
TARGET_RAW = "price"            # nightly price, LOCAL currency (see section 2.2)
TARGET_USD = "price_usd"        # nightly price, FX-normalised to USD
TARGET_MODEL = "log_price"      # natural log of price_usd — the actual regression target
RANDOM_STATE = 42
TEST_SIZE = 0.20

# Data snapshot date. The listings file is an early-2021 crawl; the latest host_since
# value in the file is 2021-02-26, so tenure features are measured against 2021-03-01.
SNAPSHOT_DATE = pd.Timestamp("2021-03-01")

RAW_S3_URI = f"s3://{BUCKET}/{PREFIX}/raw/Listings.csv"
PROCESSED_PREFIX = f"{PREFIX}/processed"

s3 = boto3.client("s3")

print("=" * 68)
print("ITI113 — Notebook 01: EDA & Feature Engineering (Regression)")
print("=" * 68)
print(f"Team        : {TEAM_ID}")
print(f"Student     : {STUDENT_ID}")
print(f"Project     : {PROJECT_NAME}")
print(f"Region      : {region}")
print(f"Role        : {role.split('/')[-1]}")
print(f"Raw data    : {RAW_S3_URI}")
print(f"Processed   : s3://{BUCKET}/{PROCESSED_PREFIX}")
print(f"Target      : {TARGET_MODEL}  (log of FX-normalised nightly price)")
print("=" * 68)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


ITI113 — Notebook 01: EDA & Feature Engineering (Regression)
Team        : team14
Student     : s1402
Project     : airbnb-pricing
Region      : ap-southeast-1
Role        : SageMakerExecutionRole-ITI113-Team14
Raw data    : s3://nyp-26s1-iti113/iti113/team14/data/airbnb-pricing/raw/Listings.csv
Processed   : s3://nyp-26s1-iti113/iti113/team14/data/airbnb-pricing/processed
Target      : log_price  (log of FX-normalised nightly price)


## 1. Load the raw dataset

The dataset is the Airbnb multi-city listings snapshot: **279,712 listings across 10 cities**
(Paris, New York, Sydney, Rome, Rio de Janeiro, Istanbul, Mexico City, Bangkok, Cape Town,
Hong Kong), 33 columns.

| Column | Type | Meaning | Modelling role |
|---|---|---|---|
| `listing_id` | int | Unique listing key | identifier — dropped |
| `name` | str | Free-text listing title | not used in v1 (NLP candidate) |
| `host_id` | int | Host key | grouping key for leakage checks |
| `host_since` | date | Date host joined | → tenure feature |
| `host_location` | str | Self-reported host location | high-cardinality free text — dropped |
| `host_response_time` | str | Ordinal response band | → ordinal encode |
| `host_response_rate` | float | 0–1 | numeric |
| `host_acceptance_rate` | float | 0–1 | numeric |
| `host_is_superhost` | t/f | Superhost badge | binary |
| `host_total_listings_count` | int | Portfolio size | → professionalisation features |
| `host_has_profile_pic` | t/f | Profile photo present | binary |
| `host_identity_verified` | t/f | ID verified | binary |
| `neighbourhood` | str | 660 distinct values | → smoothed target encoding |
| `district` | str | Sub-region | **86.8% missing — dropped** |
| `city` | str | One of 10 cities | categorical + FX key |
| `latitude` / `longitude` | float | Coordinates | → geospatial features |
| `property_type` | str | 144 distinct values | → grouped categorical |
| `room_type` | str | Entire / Private / Hotel / Shared | categorical |
| `accommodates` | int | Guest capacity | numeric |
| `bedrooms` | float | Bedroom count | numeric (10.5% missing) |
| `amenities` | JSON str | List of amenity strings | → count, flags, luxury score |
| **`price`** | **int** | **Nightly price, LOCAL currency** | **TARGET** |
| `minimum_nights` / `maximum_nights` | int | Stay policy | → regime features |
| `review_scores_*` (7 cols) | float | Guest ratings | → composites (32.8% missing) |
| `instant_bookable` | t/f | Instant book enabled | binary |

In [2]:
DTYPES = {
    "listing_id": "int64",
    "host_id": "int64",
    "price": "float64",
    "accommodates": "int16",
    "minimum_nights": "int64",
    "maximum_nights": "int64",
}

# The source file contains Latin-1 encoded characters in listing names.
obj = s3.get_object(Bucket=BUCKET, Key=f"{PREFIX}/raw/Listings.csv")
df_raw = pd.read_csv(
    io.BytesIO(obj["Body"].read()),
    encoding="latin-1",
    dtype=DTYPES,
    low_memory=False,
)

print(f"Shape        : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Memory       : {df_raw.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print(f"Cities       : {df_raw['city'].nunique()}")
print(f"Unique hosts : {df_raw['host_id'].nunique():,}")
print(f"Duplicate listing_id: {df_raw['listing_id'].duplicated().sum()}")
df_raw.head(3)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8 }                                                                                           │
│    9                                                                                             │
│   10 # The source file contains Latin-1 encoded characters in listing names.                     │
│ ❱ 11 obj = s3.get_object(Bucket=BUCKET, Key=f"{PREFIX}/raw/Listings.csv")                        │
│   12 df_raw = pd.read_csv(                                                                       │
│   13 │   io.BytesIO(obj["Body"].read()),                                                         │
│   14 │   encoding="latin-1",                                                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1094 in _make_api_call                │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1095 │   │   else:                                                                             │
│   1096 │   │   │   return parsed_response                                                        │
│   1097                                                     

**Observed result**

```text
Shape        : 279,712 rows x 33 columns
Cities       : 10
Duplicate listing_id: 0
```

The primary key is clean. Note that `host_id` is *not* unique — a single host may own many
listings. That matters for the split strategy (§6) because two listings from the same host are
not statistically independent.

## 2. Data Suitability Assessment

Before any modelling we test whether this dataset can actually support the stated objective:
*predict a continuous nightly price*. Four questions, each with a pass/fail test.

| # | Question | Test |
|---|---|---|
| S1 | Is the target present, continuous, and complete? | dtype, null rate, cardinality |
| S2 | Is the target measured on a **single consistent scale**? | per-city distribution comparison |
| S3 | Do the features carry usable signal about the target? | variance decomposition / mutual information |
| S4 | Is the sample large enough for the intended model class? | rows-per-parameter heuristic |

### 2.1 S1 — Target integrity

In [ ]:
tgt = df_raw[TARGET_RAW]

print("S1 — TARGET INTEGRITY")
print("-" * 60)
print(f"dtype                    : {tgt.dtype}")
print(f"Nulls                    : {tgt.isna().sum()} ({tgt.isna().mean():.2%})")
print(f"Distinct values          : {tgt.nunique():,}")
print(f"Zero or negative         : {(tgt <= 0).sum()}")
print(f"Min / Median / Max       : {tgt.min():,.0f} / {tgt.median():,.0f} / {tgt.max():,.0f}")
print(f"Skewness                 : {tgt.skew():.2f}")
print(f"Kurtosis                 : {tgt.kurtosis():.2f}")
print()
print("VERDICT: continuous, fully populated, high cardinality -> suitable for regression.")
print("         113 non-positive values require an explicit outlier policy (section 4.3).")

### 2.2 S2 — The scale problem: `price` is not in one currency

This is the single most consequential finding in the dataset and it is invisible unless you
look per city. `price` is recorded in each city's **local currency**. Pooling the raw column
across cities produces a target that is a mixture of ten different measurement units.

In [ ]:
city_price = (
    df_raw.groupby("city")[TARGET_RAW]
    .agg(n="size", p25=lambda s: s.quantile(0.25), median="median",
         p75=lambda s: s.quantile(0.75), p99=lambda s: s.quantile(0.99), max="max")
    .sort_values("median")
)
print("S2 — RAW `price` BY CITY (mixed currencies)")
print("-" * 78)
print(city_price.round(0).to_string())

ratio = city_price["median"].max() / city_price["median"].min()
print(f"\nRatio of largest to smallest city median: {ratio:,.1f}x")
print("A 30x spread in medians is not a real price difference — it is a unit mismatch.")

**Observed result**

```text
                median        p99         max
Rome              65.0      600.0     10571.0     <- EUR
Paris             80.0      600.0     12000.0     <- EUR
New York          99.0      850.0     10000.0     <- USD
Sydney           120.0     1721.4     28613.0     <- AUD
Istanbul         252.0     3805.7    179532.0     <- TRY
Rio de Janeiro   280.0     5693.3    625216.0     <- BRL
Hong Kong        386.0     8799.0     83665.0     <- HKD
Mexico City      661.0     7294.4    499000.0     <- MXN
Cape Town       1069.0    22018.9    180000.0     <- ZAR
Bangkok         1100.0    15552.6    300177.0     <- THB

Ratio of largest to smallest city median: 16.9x
```

**Why this fails S2.** A model trained on the pooled raw column would learn that "Bangkok is
expensive" and "Rome is cheap" — the exact inverse of reality. Worse, any global outlier rule
(for example "drop anything above \$1,000") would delete a normal Bangkok listing (1,000 THB ≈
\$32) while retaining a genuinely anomalous Rome listing at 900 EUR ≈ \$1,062. Every downstream
step — outlier trimming, scaling, loss weighting, error reporting — is corrupted by the unit
mismatch.

**Resolution.** Normalise to a single numéraire (USD) before anything else. This is applied in
§4.2 and is treated as a documented, versioned modelling assumption, not a silent transformation.

> **S2 verdict: CONDITIONAL PASS** — suitable *only after* FX normalisation.

### 2.3 S3 — Does the feature set carry signal?

A cheap, model-free test: decompose the variance of log-price into a between-group and
within-group component for each candidate predictor. If a feature carries no information,
group means are all equal and the between-group share (η², "eta squared") is ~0.

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}
        = \frac{\sum_g n_g(\bar{y}_g - \bar{y})^2}{\sum_i (y_i - \bar{y})^2}$$

η² is bounded in [0, 1] and is directly interpretable as *"the fraction of price variance this
single variable explains on its own"*.

In [ ]:
def eta_squared(y: pd.Series, g: pd.Series) -> float:
    """Fraction of variance in y explained by grouping variable g."""
    d = pd.DataFrame({"y": y, "g": g}).dropna()
    grand = d["y"].mean()
    grp = d.groupby("g")["y"].agg(["mean", "size"])
    ss_between = (grp["size"] * (grp["mean"] - grand) ** 2).sum()
    ss_total = ((d["y"] - grand) ** 2).sum()
    return ss_between / ss_total if ss_total > 0 else 0.0


# Provisional log target for the suitability check only (proper version built in section 4).
_fx_probe = {"Paris": 1.18, "New York": 1.0, "Bangkok": 0.0316, "Rio de Janeiro": 0.19,
             "Sydney": 0.75, "Istanbul": 0.12, "Rome": 1.18, "Hong Kong": 0.1287,
             "Mexico City": 0.05, "Cape Town": 0.068}
_probe = df_raw[df_raw[TARGET_RAW] > 0].copy()
_probe["_logp"] = np.log(_probe[TARGET_RAW] * _probe["city"].map(_fx_probe))

candidates = {
    "neighbourhood": _probe["neighbourhood"],
    "city": _probe["city"],
    "accommodates": _probe["accommodates"].clip(1, 12),
    "room_type": _probe["room_type"],
    "property_type": _probe["property_type"],
    "bedrooms": _probe["bedrooms"].fillna(-1).clip(-1, 8),
    "host_total_listings (binned)": pd.cut(_probe["host_total_listings_count"],
                                           [-1, 1, 2, 5, 20, 1e6]),
    "host_is_superhost": _probe["host_is_superhost"],
    "instant_bookable": _probe["instant_bookable"],
}
sig = pd.Series({k: eta_squared(_probe["_logp"], v) for k, v in candidates.items()})
sig = sig.sort_values(ascending=False)

print("S3 — SINGLE-VARIABLE EXPLANATORY POWER (eta^2 on log price)")
print("-" * 60)
for k, v in sig.items():
    bar = "#" * int(v * 60)
    print(f"{k:<30} {v:6.3f}  {bar}")

del _probe

**Observed result**

```text
neighbourhood                   0.316  ###################
property_type                   0.237  ##############
city                            0.222  #############
accommodates                    0.222  #############
bedrooms                        0.207  ############
room_type                       0.164  #########
host_total_listings (binned)    0.011
instant_bookable                0.007
host_is_superhost               0.000
```

Three things to take away:

1. **`neighbourhood` alone explains ~32% of log-price variance** — more than any other single
   variable, and more than `city` itself (0.22). Location dominates, and it dominates at a finer
   grain than city. This justifies the effort spent on the neighbourhood price index in §7, but
   also warns that 660 raw categories would explode a one-hot matrix, so an encoding strategy is
   required.
2. **Capacity (`accommodates` 0.22, `bedrooms` 0.21) and product type are the next tier.**
   Together with location these are the classic hedonic price drivers. Note that
   `property_type` (0.24) scores above `room_type` (0.16) because its 144 levels encode capacity
   and room type jointly — a warning that raw η² rewards cardinality and must be read alongside
   the level counts.
3. **`host_is_superhost` has η² ≈ 0.000.** The "superhost premium" that hosts widely believe in
   is *not present* in this data at the population level. We return to this in §3.6 — it turns
   out to be a genuine finding, not a data error.

> **S3 verdict: PASS** — ample signal, concentrated in location and capacity.

### 2.4 S4 — Sample adequacy and suitability verdict

In [ ]:
n_rows = len(df_raw)
n_planned_features = 60          # approximate width of the engineered matrix
n_train = int(n_rows * (1 - TEST_SIZE))

print("S4 — SAMPLE ADEQUACY")
print("-" * 60)
print(f"Total rows                     : {n_rows:,}")
print(f"Training rows (80%)            : {n_train:,}")
print(f"Planned engineered features    : ~{n_planned_features}")
print(f"Rows per feature               : {n_train / n_planned_features:,.0f}")
print(f"Smallest city (Hong Kong)      : {df_raw['city'].value_counts().min():,}")
print(f"Smallest neighbourhood         : {df_raw['neighbourhood'].value_counts().min()}")
print()
print("Heuristic: >=50 rows/feature is comfortable for regularised linear models;")
print("gradient boosting is comfortable well above that. Both are satisfied.")
print("Caveat: rare neighbourhoods need shrinkage, not raw category means (section 5.9).")

### Suitability summary

| Test | Result | Action taken |
|---|---|---|
| S1 — target integrity | **PASS** | 113 non-positive prices routed to the outlier policy |
| S2 — single scale | **CONDITIONAL** | FX normalisation to USD applied in §4.2 before any other step |
| S3 — feature signal | **PASS** | Location and capacity carry strong signal; superhost does not |
| S4 — sample size | **PASS** | ~3,700 training rows per feature; shrinkage needed for rare neighbourhoods |

**Overall: the dataset is suitable for nightly-price regression, conditional on currency
normalisation.** Two structural limitations are recorded now and carried into the final report:

- There is **no temporal dimension** — one snapshot per listing, no calendar, no seasonality, no
  occupancy. The model can therefore only ever predict a *listing's standing rate*, never a
  demand-responsive or date-specific price.
- `price` is the **asking price set by the host**, not a transacted price. The model learns to
  reproduce host pricing behaviour, including any collective mispricing. This is a material
  governance point (see §7 and the final report).

## 3. Exploratory Data Analysis

The EDA below is organised around the target. Each subsection ends with a stated implication
for modelling, so that nothing is plotted without a decision attached to it.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titleweight": "bold",
})
PALETTE = ["#2E5EAA", "#D1495B", "#EDAE49", "#00798C", "#66A182",
           "#8D6A9F", "#F79256", "#5B5F97", "#A44A3F", "#7BA05B"]
print("Plot defaults set.")

### 3.1 Missingness structure — and why it is *not* random

Missing values here are informative. Treating them as random noise and imputing silently would
destroy signal and introduce bias.

In [ ]:
miss = (df_raw.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]

fig, ax = plt.subplots(figsize=(7.5, 5))
colors = ["#D1495B" if v > 50 else "#EDAE49" if v > 20 else "#2E5EAA" for v in miss.values]
ax.barh(miss.index[::-1], miss.values[::-1], color=colors[::-1])
ax.set_xlabel("% missing")
ax.set_title("Missingness by column")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
for i, v in enumerate(miss.values[::-1]):
    ax.text(v + 0.7, i, f"{v:.1f}", va="center", fontsize=7.5)
plt.tight_layout(); plt.show()

# Is review missingness random? Compare listings with and without reviews.
has_rev = df_raw["review_scores_rating"].notna()
print("\nAre listings WITHOUT reviews systematically different?")
print("-" * 62)
comp = pd.DataFrame({
    "with_reviews": df_raw[has_rev].agg({"price": "median", "accommodates": "mean",
                                         "minimum_nights": "median"}),
    "no_reviews": df_raw[~has_rev].agg({"price": "median", "accommodates": "mean",
                                        "minimum_nights": "median"}),
})
print(comp.round(2).to_string())
print(f"\nHost tenure (median days since joining):")
hs = pd.to_datetime(df_raw["host_since"], errors="coerce")
ten = (SNAPSHOT_DATE - hs).dt.days
print(f"  with reviews : {ten[has_rev].median():,.0f} days")
print(f"  no reviews   : {ten[~has_rev].median():,.0f} days")
print(f"\nShare of listings with no reviews: {(~has_rev).mean():.1%}")

**Observed result and interpretation**

```text
district              86.8%   -> structurally absent for 8 of 10 cities. DROP.
host_response_rate    46.0%   -> host has never been messaged / no data. MNAR.
host_response_time    46.0%
host_acceptance_rate  40.4%
review_scores_* (7)   32.7%   -> listing has never been reviewed. MNAR.
bedrooms              10.5%   -> studios and rooms where the field is not applicable.
host_since etc.        0.06%  -> 165 orphaned listings with a deleted host record.
```

**These are three different mechanisms and each needs a different treatment:**

| Pattern | Mechanism | Correct treatment |
|---|---|---|
| `district` | Missing by design (not collected outside 2 cities) | **Drop the column.** Imputation would fabricate structure. |
| `review_scores_*`, `host_response_*` | **MNAR** — missing *because the listing is new or has never been contacted*. Median host tenure is 1,927 days for reviewed listings vs 1,654 for unreviewed — a 273-day gap. | Impute **and** add a `has_reviews` / `has_response_data` indicator. The indicator is the real feature; the imputed value is a placeholder. |
| `bedrooms` | Field not applicable (studio) or not filled in | Impute from the `(city, accommodates)` group median, plus a `bedrooms_missing` flag. |

Dropping the 32.7% of rows without reviews would be the naive alternative. It would be wrong:
those listings are not a random third of the market, they are the *new-listing cohort* — exactly
the population a pricing tool most needs to serve. Removing them would produce a model that
silently fails for every new host.

### 3.2 Target distribution — the case for a log transform

In [ ]:
FX_PROBE = {"Paris": 1.18, "New York": 1.0, "Bangkok": 0.0316, "Rio de Janeiro": 0.19,
            "Sydney": 0.75, "Istanbul": 0.12, "Rome": 1.18, "Hong Kong": 0.1287,
            "Mexico City": 0.05, "Cape Town": 0.068}
p_usd = (df_raw[TARGET_RAW] * df_raw["city"].map(FX_PROBE))
p_pos = p_usd[p_usd > 0]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

axes[0].hist(p_pos, bins=200, range=(0, 1000), color="#2E5EAA", edgecolor="none")
axes[0].set_title("Nightly price (USD) — raw")
axes[0].set_xlabel("USD"); axes[0].set_ylabel("listings")

axes[1].hist(np.log(p_pos), bins=100, color="#00798C", edgecolor="none")
axes[1].set_title("log(price) — near-Gaussian")
axes[1].set_xlabel("log USD")

from scipy import stats
stats.probplot(np.log(p_pos.sample(20000, random_state=RANDOM_STATE)), dist="norm", plot=axes[2])
axes[2].set_title("Q-Q plot of log(price)")
axes[2].get_lines()[0].set_markersize(2)
axes[2].get_lines()[0].set_color("#00798C")
plt.tight_layout(); plt.show()

print("TARGET SHAPE")
print("-" * 52)
print(f"{'':<12}{'raw USD':>14}{'log USD':>14}")
print(f"{'skewness':<12}{p_pos.skew():>14.2f}{np.log(p_pos).skew():>14.2f}")
print(f"{'kurtosis':<12}{p_pos.kurtosis():>14.2f}{np.log(p_pos).kurtosis():>14.2f}")
print(f"{'mean':<12}{p_pos.mean():>14.2f}{np.log(p_pos).mean():>14.2f}")
print(f"{'median':<12}{p_pos.median():>14.2f}{np.log(p_pos).median():>14.2f}")
print(f"\nMean/median ratio (raw): {p_pos.mean() / p_pos.median():.2f}  "
      f"-> strong right tail")

**Observed result**

```text
                   raw USD       log USD
skewness             91.94          0.42
kurtosis          20154.18          1.22
mean                123.33          4.27
median               71.00          4.26
```

#### Justification for modelling `log(price)` rather than `price`

**1. Distributional.** Skewness falls from **91.9 to 0.42** and excess kurtosis from **20,154 to
1.22**. Ordinary least squares and squared-error boosting both assume approximately symmetric,
finite-variance residuals; the raw target violates this so severely that a handful of listings
would dominate the entire loss.

**2. Error structure.** Price errors are naturally *multiplicative*, not additive. Being \$40
wrong on a \$50 room is a catastrophic recommendation; being \$40 wrong on a \$900 villa is
noise. Minimising squared error on the raw scale weights those identically. Minimising it on the
log scale is equivalent to minimising *relative* error, because

$$\log \hat{y} - \log y = \log\frac{\hat{y}}{y}$$

so a residual of $0.10$ means "10% off" regardless of the price level. This aligns the loss
function with how the error is actually experienced by a host.

**3. Functional form.** Hedonic price theory models value as a *product* of attribute premia
(location factor × size factor × quality factor). Taking logs turns that product into a sum,
which is exactly the additive form linear models and regression trees fit natively:

$$y = \beta_0 \prod_j x_j^{\beta_j} \;\;\Longleftrightarrow\;\; \log y = \log\beta_0 + \sum_j \beta_j \log x_j$$

**4. Positivity.** $\exp(\cdot)$ of any prediction is strictly positive, so back-transformed
predictions can never be negative — which a linear model on the raw scale absolutely would be
for cheap listings.

> **Decision: the regression target is `log_price = ln(price_usd)`.** All model-selection
> metrics are computed on the log scale; all *business-facing* metrics are reported in USD after
> back-transformation. Both are logged to MLflow in Notebook 02.

### 3.3 Pricing anomalies

Three distinct anomaly classes exist and they need different handling.

In [ ]:
probe = df_raw.copy()
probe["price_usd"] = p_usd

print("ANOMALY CLASS 1 — impossible prices")
print("-" * 62)
print(f"price == 0                       : {(probe['price_usd'] == 0).sum()}")
print(f"price < $5 USD                   : {(probe['price_usd'] < 5).sum()}")
print(f"price > $2,000 USD               : {(probe['price_usd'] > 2000).sum()}")
print(f"price > $10,000 USD              : {(probe['price_usd'] > 10000).sum()}")
print(f"Max                              : ${probe['price_usd'].max():,.0f} "
      f"({probe.loc[probe['price_usd'].idxmax(), 'city']})")

print("\nANOMALY CLASS 2 — economically implausible combinations")
print("-" * 62)
weird = probe[(probe["accommodates"] >= 8) & (probe["price_usd"] < 15)]
print(f"Sleeps 8+ but costs < $15/night  : {len(weird)}")
print(weird[["city", "room_type", "accommodates", "bedrooms", "price", "price_usd"]]
      .head(5).to_string(index=False))

print("\nANOMALY CLASS 3 — sentinel values in stay policy")
print("-" * 62)
print(f"minimum_nights > 365             : {(probe['minimum_nights'] > 365).sum()}"
      f"   (max = {probe['minimum_nights'].max():,})")
print(f"maximum_nights > 1125            : {(probe['maximum_nights'] > 1125).sum()}"
      f"   (max = {probe['maximum_nights'].max():,})")
print(f"maximum_nights == 1125 exactly   : {(probe['maximum_nights'] == 1125).sum():,}"
      f"   <- Airbnb platform default, not a host choice")
print(f"minimum_nights >= 30             : {(probe['minimum_nights'] >= 30).sum():,}"
      f"   <- monthly-rental regime")

**Observed result and treatment plan**

```text
price == 0                       : 113
price > $2,000 USD               : 660
Max                              : $118,791 (Rio de Janeiro)
Sleeps 8+ but costs < $15/night  : 51
maximum_nights == 1125 exactly   : ~180,000  <- platform default
minimum_nights >= 30             : 33,000+   <- different pricing regime
```

| Class | Example | Diagnosis | Treatment |
|---|---|---|---|
| **1. Impossible prices** | \$0/night; \$118,791/night in Rio | Data-entry error, or a host parking a listing with a deterrent price | Per-city quantile trimming at [0.5%, 99.5%] — §4.3 |
| **2. Implausible combinations** | 16-guest entire home at \$10/night in Bangkok | Currency field mis-entry or a per-person price entered as total | Mostly removed by class-1 trimming; the residue is **irreducible label noise** and is quantified in the error analysis (Notebook 02, §9.9) |
| **3. Sentinel values** | `maximum_nights = 2,147,483,647` (int32 max) | Platform defaults and overflow, not host intent | Clip to the platform's real bounds; add `is_monthly_only` to capture the genuine regime change at 30 nights |

The `minimum_nights >= 30` group deserves emphasis: those listings are competing in the
**monthly-rental market**, not the nightly market. Their per-night rate is set by a different
economic process (they undercut on a nightly basis to win a long booking). Median price for that
group is \$87 versus \$71 overall — they are not cheaper, they are *different*. A binary regime
flag lets the model learn a separate intercept rather than averaging the two markets together.

### 3.4 Spatial structure — distance decay is not monotonic

In [ ]:
CITY_CENTRE = {
    "Paris": (48.8566, 2.3522), "New York": (40.7580, -73.9855),
    "Bangkok": (13.7563, 100.5018), "Rio de Janeiro": (-22.9068, -43.1729),
    "Sydney": (-33.8688, 151.2093), "Istanbul": (41.0082, 28.9784),
    "Rome": (41.9028, 12.4964), "Hong Kong": (22.3193, 114.1694),
    "Mexico City": (19.4326, -99.1332), "Cape Town": (-33.9249, 18.4241),
}


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two coordinate arrays."""
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


probe["dist_km"] = haversine_km(
    probe["latitude"].values, probe["longitude"].values,
    probe["city"].map(lambda c: CITY_CENTRE[c][0]).values,
    probe["city"].map(lambda c: CITY_CENTRE[c][1]).values,
)

bands = [0, 1, 2, 3, 5, 8, 12, 20, 1000]
probe["band"] = pd.cut(probe["dist_km"], bands)
decay = probe[probe["price_usd"] > 0].groupby("band", observed=True).agg(
    n=("price_usd", "size"), median_usd=("price_usd", "median"))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(range(len(decay)), decay["median_usd"], "o-", color="#D1495B", lw=2)
axes[0].set_xticks(range(len(decay)))
axes[0].set_xticklabels([str(i) for i in decay.index], rotation=40, ha="right", fontsize=7.5)
axes[0].set_ylabel("median price (USD)")
axes[0].set_title("Price vs distance from city centre — pooled")
axes[0].set_xlabel("distance band (km)")

for i, c in enumerate(["Paris", "New York", "Cape Town", "Sydney"]):
    sub = probe[(probe["city"] == c) & (probe["price_usd"] > 0)]
    d = sub.groupby(pd.cut(sub["dist_km"], bands), observed=True)["price_usd"].median()
    axes[1].plot(range(len(d)), d.values, "o-", label=c, color=PALETTE[i], lw=1.6, ms=4)
axes[1].set_xticks(range(len(bands) - 1))
axes[1].set_xticklabels([f"{bands[i]}-{bands[i+1]}" for i in range(len(bands) - 1)],
                        rotation=40, ha="right", fontsize=7.5)
axes[1].set_title("Distance decay differs by city")
axes[1].set_xlabel("distance band (km)"); axes[1].legend(fontsize=7.5)
plt.tight_layout(); plt.show()

print(decay.round(1).to_string())

**Observed result**

```text
band          n        median_usd
(0, 1]     16,918          86.1
(1, 2]     29,910          94.4   <- PEAK, not the centre
(2, 3]     33,079          81.6
(3, 5]     70,143          74.3
(5, 8]     51,525          62.7
(8, 12]    35,007          59.0
(12, 20]   22,720          55.0
(20, 1000] 17,683          64.6   <- rises again
```

**Two non-obvious findings:**

1. **The price peak is 1–2 km out, not at the centroid.** City centroids in these cities are
   commercial/administrative districts (Châtelet, Times Square, the Colosseum) with hotels and
   offices rather than desirable residential stock. The premium residential belt sits just
   outside it.
2. **Price rises again beyond 20 km.** These are coastal and resort properties — Cape Town's
   Atlantic Seaboard, Sydney's northern beaches, Rio's Barra. "Far from the centre" and "cheap"
   are the same thing only in the middle of the distribution.

**Implication for modelling.** The relationship between distance and price is **non-monotonic**,
so a linear coefficient on distance is misspecified (this is confirmed in Notebook 02: Ridge
reaches R² 0.64 while gradient boosting reaches 0.75). Two responses are needed:

- keep raw `latitude`/`longitude` so tree models can learn arbitrary spatial partitions, and
- add a **neighbourhood price index** (§5.9) that encodes local price level directly rather than
  forcing the model to reconstruct it from coordinates.

### 3.5 Simpson's paradox: pooled correlations are actively misleading

This is the most important methodological finding in the EDA. Several features have a
correlation with price whose **sign flips** when you condition on city.

In [ ]:
probe_pos = probe[probe["price_usd"] > 0].copy()
probe_pos["logp"] = np.log(probe_pos["price_usd"])
probe_pos["is_pro_host"] = (probe_pos["host_total_listings_count"] > 5).astype(int)
probe_pos["superhost"] = (probe_pos["host_is_superhost"] == "t").astype(int)
probe_pos["n_amenities"] = probe_pos["amenities"].fillna("[]").str.count(",") + 1

feats = ["is_pro_host", "superhost", "n_amenities", "accommodates"]
rows = []
for f in feats:
    pooled = probe_pos[f].corr(probe_pos["logp"])
    within = probe_pos.groupby("city").apply(lambda g: g[f].corr(g["logp"]))
    rows.append({"feature": f, "pooled_r": round(pooled, 3),
                 "within_city_min": round(within.min(), 3),
                 "within_city_median": round(within.median(), 3),
                 "within_city_max": round(within.max(), 3),
                 "sign_flip": "YES" if pooled * within.median() < 0 else "no"})
print("POOLED vs WITHIN-CITY CORRELATION WITH log(price)")
print("-" * 90)
print(pd.DataFrame(rows).to_string(index=False))

print("\n\nWHY: city-level confounding")
print("-" * 90)
conf = probe_pos.groupby("city").agg(
    pro_host_share=("is_pro_host", "mean"),
    median_price_usd=("price_usd", "median"),
    n=("price_usd", "size")).sort_values("pro_host_share", ascending=False)
print(conf.round(3).to_string())
print(f"\nCorrelation between a city's professional-host share and its median price: "
      f"{conf['pro_host_share'].corr(conf['median_price_usd']):.3f}")

**Observed result**

```text
feature        pooled_r   within_city_median   sign_flip
is_pro_host      -0.013               +0.16        YES
n_amenities      +0.126               +0.30         no (but ~2.4x understated)
superhost        -0.019               +0.03        YES
accommodates     +0.434               +0.51         no

city              pro_host_share   median_price_usd
Hong Kong                  0.608              49.68
Bangkok                    0.430              34.76
Rome                       0.253              76.70
Cape Town                  0.236              72.62
Istanbul                   0.239              30.24
Mexico City                0.235              33.05
Rio de Janeiro             0.185              53.20
New York                   0.135              99.00
Sydney                     0.134              90.00
Paris                      0.109              94.40
```

**The mechanism.** Cities with the *highest* share of professional multi-listing hosts (Hong
Kong 61%, Bangkok 43%) are also the cities with the *lowest* USD price levels. Pooling the data
therefore produces a spurious negative association: professional hosts look cheap, when in fact
*within any given city* a professional host charges materially more — in Paris the median rises
from \$88.50 for single-listing hosts to \$145.10 for hosts with 6–20 listings, a 64% gap. Exactly the same
confounding suppresses the amenity-count correlation from +0.30 down to +0.13.

**Implications, which drive three concrete design decisions:**

1. **`city` must enter the model as a feature**, not be averaged away. It is the confounder that
   blocks the back-door path from every host-level attribute to price.
2. **Feature screening must be done within city.** A pooled-correlation filter would have
   discarded `is_pro_host` as noise.
3. **The train/test split must be stratified by city** (§6) so that city composition is
   identical across splits and per-city error is measurable.

### 3.6 The superhost premium is a myth — and the amenity premium is a proxy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sh = probe_pos.groupby(["city", "superhost"])["price_usd"].median().unstack()
sh["premium_%"] = (sh[1] / sh[0] - 1) * 100
sh = sh.sort_values("premium_%")
cols = ["#D1495B" if v < 0 else "#00798C" for v in sh["premium_%"]]
axes[0].barh(sh.index, sh["premium_%"], color=cols)
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_xlabel("Superhost median price premium (%)")
axes[0].set_title("Superhost 'premium' by city — sign varies")

AM_TEST = ["Pool", "Gym", "Dishwasher", "Heating", "Air conditioning",
           "Free parking on premises", "Elevator", "Bathtub"]
am_series = probe_pos["amenities"].fillna("[]")
lift = {}
for a in AM_TEST:
    has = am_series.str.contains(f'"{a}"', regex=False)
    lift[a] = (probe_pos.loc[has, "price_usd"].median()
               / probe_pos.loc[~has, "price_usd"].median() - 1) * 100
lift = pd.Series(lift).sort_values()
cols = ["#D1495B" if v < 0 else "#00798C" for v in lift.values]
axes[1].barh(lift.index, lift.values, color=cols)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_xlabel("Median price lift (%)")
axes[1].set_title("Naive amenity 'lift' — mostly climate/city proxies")
plt.tight_layout(); plt.show()

print("Superhost premium by city (%):")
print(sh["premium_%"].round(1).to_string())
print("\nNaive amenity lift (%):")
print(lift.round(1).to_string())

**Observed result**

```text
Superhost premium by city (%)      Naive amenity lift (%)
Rio de Janeiro   -30.0             Gym                 -12.5
Hong Kong        -29.3             Free parking         -8.3
Cape Town         -8.6             Pool                 -0.5
Bangkok           -4.5             Air conditioning     +5.9
New York           0.0             Elevator             +5.9
Rome              +3.1             Bathtub             +68.6
Sydney            +8.3             Heating             +69.4
Paris            +15.0             Dishwasher          +70.2
Istanbul         +18.8
Mexico City      +25.7
```

**Superhost.** The badge is associated with a *lower* median price in Rio (−30%) and Hong Kong
(−29%) but a *higher* one in Mexico City (+26%) and Paris (+15%). Pooled η² was 0.000 because
these opposing effects cancel. The plausible reading: superhost status is earned through
occupancy and responsiveness, and in some markets the fastest route to occupancy is to price
*below* the market. The badge is a proxy for operational behaviour, not for property quality.
It is retained as a feature but should not be interpreted causally, and it should never be used
to justify a price recommendation to a host.

**Amenities.** "Heating" appears to add 69% to price. It does not. Heating is near-universal in
Paris, New York and Rome (cold, expensive cities) and near-absent in Bangkok and Rio (warm,
cheap cities). The coefficient is measuring **climate and city**, not the value of a radiator.
The same logic explains why "Pool" shows a *negative* lift: pools are standard in cheap tropical
markets and rare in expensive temperate ones.

> **Consequence for feature engineering.** Raw amenity flags are contaminated by city. We keep
> them (a tree model can interact them with `city` and recover the within-city effect) but we do
> **not** interpret their marginal effects, and we add a `luxury_score` aggregate (§5.4) that is
> more robust than any single flag.

### 3.7 A genuine pricing signal hidden in the review scores

`review_scores_value` asks guests whether the listing was good *value for money*. If hosts
systematically over-price, this should decline as price rises — and it does, monotonically.

In [ ]:
rev = probe_pos[probe_pos["review_scores_rating"].notna()].copy()
rev["value_gap"] = rev["review_scores_value"] - rev["review_scores_rating"] / 10.0
rev["price_quintile"] = rev.groupby("city")["price_usd"].transform(
    lambda s: pd.qcut(s, 5, labels=False, duplicates="drop"))

g = rev.groupby("price_quintile").agg(
    n=("value_gap", "size"),
    mean_overall_rating=("review_scores_rating", "mean"),
    mean_value_score=("review_scores_value", "mean"),
    mean_value_gap=("value_gap", "mean"))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(g.index, g["mean_value_gap"], color=["#00798C", "#66A182", "#EDAE49", "#F79256", "#D1495B"])
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(5))
ax.set_xticklabels(["Q1\ncheapest", "Q2", "Q3", "Q4", "Q5\npriciest"])
ax.set_ylabel("mean value gap")
ax.set_title("Perceived value falls as price rises (within-city quintiles)")
plt.tight_layout(); plt.show()

print(g.round(3).to_string())
print("\nNote: overall rating RISES with price (92.3 -> 94.4) while the value")
print("      score is FLAT (9.33 -> 9.31). Guests acknowledge the quality but")
print("      not the price. The gap between them is the signal.")

**Observed result**

```text
price_quintile   mean_overall_rating   mean_value_score   mean_value_gap
Q1 (cheapest)             92.33               9.331            +0.095
Q2                        93.05               9.347            +0.041
Q3                        93.45               9.350            +0.003
Q4                        93.97               9.347            -0.051
Q5 (priciest)             94.37               9.313            -0.126
```

This is a clean, monotone effect and it is *not* an artefact of quality. Overall rating **rises**
with price (92.3 → 94.4, so expensive listings genuinely are better), but the value score stays
flat at ~9.33. Guests recognise higher quality and refuse to credit it as better value.

Define the **value gap**:

$$\text{value\_gap} = s_{\text{value}} - \frac{s_{\text{overall}}}{10}$$

Both terms are rescaled to a 0–10 range, so the gap is *"how much better or worse the value
verdict is than the overall verdict"*. A negative gap means guests thought the listing was good
but over-priced.

**Two things follow.** First, `value_gap` is a legitimately useful predictor — it is one of the
few variables that carries information about a listing's *position relative to its own market*
rather than its absolute attributes. Second, and importantly for governance: `value_gap` is
**measured after the price was set**, so it is downstream of the target. It is safe for a
descriptive/estimation model but it must be excluded from any counterfactual use ("what if I
raised my price to X?"), because it would encode the answer. This restriction is recorded in the
model card in the final report.

### 3.8 Correlation structure

In [ ]:
num_probe = pd.DataFrame({
    "log_price": probe_pos["logp"],
    "accommodates": probe_pos["accommodates"],
    "bedrooms": probe_pos["bedrooms"],
    "n_amenities": probe_pos["n_amenities"],
    "dist_km": probe_pos["dist_km"],
    "host_listings": np.log1p(probe_pos["host_total_listings_count"]),
    "min_nights": np.log1p(probe_pos["minimum_nights"].clip(1, 365)),
    "rating": probe_pos["review_scores_rating"],
    "cleanliness": probe_pos["review_scores_cleanliness"],
    "location_score": probe_pos["review_scores_location"],
    "resp_rate": probe_pos["host_response_rate"],
})
corr = num_probe.corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
im = axes[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_xticks(range(len(corr))); axes[0].set_xticklabels(corr.columns, rotation=55, ha="right", fontsize=7.5)
axes[0].set_yticks(range(len(corr))); axes[0].set_yticklabels(corr.columns, fontsize=7.5)
axes[0].set_title("Pearson correlation (pooled)")
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[0].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                     fontsize=6, color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")
plt.colorbar(im, ax=axes[0], fraction=0.046)

tc = corr["log_price"].drop("log_price").sort_values()
axes[1].barh(tc.index, tc.values,
             color=["#D1495B" if v < 0 else "#2E5EAA" for v in tc.values])
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("Correlation with log(price)")
axes[1].set_xlabel("Pearson r")
plt.tight_layout(); plt.show()

print("Max absolute off-diagonal correlation between predictors:")
c2 = corr.drop("log_price").drop(columns="log_price").abs()
np.fill_diagonal(c2.values, 0)
print(f"  {c2.max().max():.3f} "
      f"({c2.max().idxmax()} <-> {c2[c2.max().idxmax()].idxmax()})")

**Observed result and interpretation**

The strongest single linear predictor of `log_price` is `accommodates` (r = 0.43), followed by
`bedrooms` (0.38). Everything else is below |0.20|. Two consequences:

- **No single feature is close to sufficient.** The maximum |r| of 0.43 implies ~19% of variance
  from the best single predictor, versus 74% achieved by the final model — so nearly all of the
  performance comes from *interactions and non-linearity*, which is a strong prior in favour of
  tree ensembles over linear models. Notebook 02 confirms exactly this.
- **`accommodates` and `bedrooms` are correlated at 0.72.** Not high enough to force a drop,
  but high enough that their *ratio* carries information neither carries alone — the basis for
  the `persons_per_bedroom` feature in §5.5.
- The review sub-scores are mutually correlated at ~0.60 on average, which is why we collapse them into a
  composite plus two *differences* (§5.7) rather than feeding seven near-duplicate columns.

## 4. Data Cleaning & Transformation

Cleaning is applied in a fixed order, because several steps depend on earlier ones. In
particular **currency normalisation must precede outlier trimming**, otherwise the trim
thresholds are meaningless (see §2.2).

| Step | Operation | Rows/cols affected |
|---|---|---|
| 4.1 | Drop structurally unusable columns | 4 columns |
| 4.2 | FX normalisation to USD | new column |
| 4.3 | Per-city outlier trimming | ~2,840 rows (1.0%) |
| 4.4 | Sentinel-value repair on stay policy | ~2 columns |
| 4.5 | Missing-value strategy | 12 columns |

### 4.1 Drop structurally unusable columns

In [ ]:
df = df_raw.copy()

DROP_COLS = {
    "district": "86.8% missing — collected for only 2 of 10 cities",
    "host_location": "free text, 10k+ distinct values, largely redundant with `city`",
    "name": "free text; reserved for a future NLP feature set, not used in v1",
    "listing_id": "identifier — no predictive content, retained separately for traceability",
}
LISTING_IDS = df["listing_id"].copy()          # keep for audit trail
HOST_IDS = df["host_id"].copy()                # keep for group-leakage checks

for c, reason in DROP_COLS.items():
    print(f"DROP {c:<16} : {reason}")
df = df.drop(columns=list(DROP_COLS.keys()))
print(f"\nColumns: {df_raw.shape[1]} -> {df.shape[1]}")

### 4.2 Currency normalisation — a documented, versioned assumption

Every price is multiplied by a fixed local-currency-to-USD rate. Because these rates are an
**analyst-supplied input, not data**, they are treated as a first-class artefact: versioned,
printed, saved to S3, and logged to MLflow in Notebook 02 so that any future re-run can be
reconciled against the exact rates used.

Rates are approximate mid-market rates for **Q1 2021**, matching the vintage of the snapshot
(latest `host_since` = 2021-02-26).

In [ ]:
FX_VERSION = "fx_2021Q1_v1"
FX_RATES = {
    # city            local -> USD    currency
    "Paris":          1.1800,   # EUR
    "Rome":           1.1800,   # EUR
    "New York":       1.0000,   # USD
    "Sydney":         0.7500,   # AUD
    "Hong Kong":      0.1287,   # HKD
    "Istanbul":       0.1200,   # TRY
    "Rio de Janeiro": 0.1900,   # BRL
    "Bangkok":        0.0316,   # THB
    "Mexico City":    0.0500,   # MXN
    "Cape Town":      0.0680,   # ZAR
}
FX_CURRENCY = {"Paris": "EUR", "Rome": "EUR", "New York": "USD", "Sydney": "AUD",
               "Hong Kong": "HKD", "Istanbul": "TRY", "Rio de Janeiro": "BRL",
               "Bangkok": "THB", "Mexico City": "MXN", "Cape Town": "ZAR"}

missing_fx = set(df["city"].unique()) - set(FX_RATES)
assert not missing_fx, f"No FX rate defined for: {missing_fx}"

df[TARGET_USD] = df[TARGET_RAW] * df["city"].map(FX_RATES)
df["local_currency"] = df["city"].map(FX_CURRENCY)

print(f"FX table version: {FX_VERSION}")
print("-" * 72)
chk = df.groupby(["city", "local_currency"]).agg(
    median_local=(TARGET_RAW, "median"), median_usd=(TARGET_USD, "median")
).sort_values("median_usd")
print(chk.round(2).to_string())
print("\nAfter normalisation the city medians span $30-$99 — a plausible")
print("cost-of-living spread rather than a 17x unit artefact.")

# Persist the assumption alongside the data.
fx_artifact = {"fx_version": FX_VERSION, "base_currency": "USD",
               "as_of": "2021-Q1", "rates": FX_RATES, "currency": FX_CURRENCY,
               "source": "analyst-supplied mid-market rates; see model card"}
s3.put_object(Bucket=BUCKET, Key=f"{PROCESSED_PREFIX}/fx_rates.json",
              Body=json.dumps(fx_artifact, indent=2))
print(f"\nFX assumption saved: s3://{BUCKET}/{PROCESSED_PREFIX}/fx_rates.json")

### 4.3 Outlier policy — per-city quantile trimming

**The rule.** Within each city, drop listings below the 0.5th and above the 99.5th percentile of
`price_usd`.

**Why per city rather than global.** A global rule cannot separate "expensive" from "anomalous".
The 99.5th percentile is \$708 in Paris but \$1,497 in Cape Town — a \$1,000 listing is routine
in one and extreme in the other. Applying the threshold within city keeps the *relative* rarity
constant.

**Why quantiles rather than z-scores or IQR fences.** The raw distribution has excess kurtosis
of 20,154. Both the mean and the standard deviation used by a z-score rule are themselves
destroyed by the outliers they are meant to detect. Quantiles are order statistics and are
robust by construction.

**Why trim rather than winsorise.** These are not extreme-but-real observations; a \$118,791
nightly rate in Rio is a data error or a deliberate deterrent listing. Winsorising would pull it
to the 99.5th percentile and keep a fabricated observation in the training set. Deleting ~1% and
documenting it is more honest. The trimmed range is recorded as an explicit **applicability
boundary** of the model in the final report: the model is not valid outside it.

**Why 0.5%/99.5% specifically.** It removes the 113 zero-price rows and the visually obvious
tail while costing ~1% of the data. Sensitivity to this choice is checked below.

In [ ]:
n_before = len(df)

df = df[df[TARGET_USD] > 0].copy()
print(f"Removed {n_before - len(df)} non-positive prices")

lo = df.groupby("city")[TARGET_USD].transform(lambda s: s.quantile(0.005))
hi = df.groupby("city")[TARGET_USD].transform(lambda s: s.quantile(0.995))
keep = (df[TARGET_USD] >= lo) & (df[TARGET_USD] <= hi)

bounds = df.assign(lo=lo, hi=hi).groupby("city")[["lo", "hi"]].first()
bounds["removed"] = df.loc[~keep].groupby("city").size().reindex(bounds.index).fillna(0).astype(int)
print("\nPER-CITY RETAINED RANGE (USD)")
print("-" * 52)
print(bounds.round(1).to_string())

df = df[keep].copy()
print(f"\nRows: {n_before:,} -> {len(df):,} "
      f"({(n_before - len(df)) / n_before:.2%} removed)")

# Sensitivity check: how much does the choice of cut point matter?
print("\nSENSITIVITY OF THE TRIM THRESHOLD")
print("-" * 52)
_p = df_raw[TARGET_RAW] * df_raw["city"].map(FX_RATES)
_p = _p[_p > 0]
_c = df_raw.loc[_p.index, "city"]
for q in [0.001, 0.005, 0.010, 0.025]:
    l = _p.groupby(_c).transform(lambda s: s.quantile(q))
    h = _p.groupby(_c).transform(lambda s: s.quantile(1 - q))
    k = (_p >= l) & (_p <= h)
    print(f"  q={q:<6} keeps {k.mean():6.2%}  sd(log price) = {np.log(_p[k]).std():.4f}")
print("\nBeyond q=0.005 the log-scale spread barely moves: the extra rows removed")
print("are ordinary listings, not anomalies. q=0.005 is the elbow.")
del _p, _c

**Observed result**

```text
Removed 113 non-positive prices

PER-CITY RETAINED RANGE (USD)
                    lo        hi   removed
Bangkok            9.7    1003.5       186
Cape Town         13.6    2070.7       182
Hong Kong         10.6    1378.9        66
Istanbul           7.6     823.2       240
Mexico City       10.0     542.0       194
New York          23.0    1200.0       353
Paris             24.8    1009.9       620
Rio de Janeiro     9.5    1900.0       188
Rome              17.7    1178.8       262
Sydney            18.0    1725.0       323

Rows: 279,712 -> 276,985 (0.97% removed)

q=0.001  keeps 99.82%  sd(log price) = 0.8935
q=0.005  keeps 99.07%  sd(log price) = 0.8674
q=0.010  keeps 98.18%  sd(log price) = 0.8453
q=0.025  keeps 95.53%  sd(log price) = 0.8023
```

Note how different the upper bounds are: \$542 in Mexico City versus \$2,071 in Cape Town. Any
single global threshold would have been simultaneously too strict for one market and too loose
for another — the per-city rule is doing real work.

The sensitivity table is the justification for stopping at 0.5%: moving from 0.1% to 0.5% removes
0.75% of rows and cuts the log-scale spread by 0.026, whereas moving from 0.5% to 2.5% removes a
further 3.5% of rows for a similar reduction (0.065) — that second tranche is real market
variation being deleted, not noise. The marginal value per row removed collapses after 0.5%.

> **Applicability boundary (carried to the model card):** the model is validated only for nightly
> rates within each city's retained range above. Predictions outside it are extrapolation and
> should be suppressed or flagged in production.

### 4.4 Sentinel-value repair

In [ ]:
print("BEFORE")
print(f"  minimum_nights  max = {df['minimum_nights'].max():,}")
print(f"  maximum_nights  max = {df['maximum_nights'].max():,}"
      f"   (int32 overflow sentinel)")

# 365 is the practical ceiling for a "minimum stay"; 1125 is the Airbnb platform maximum.
df["minimum_nights"] = df["minimum_nights"].clip(1, 365)
df["maximum_nights"] = df["maximum_nights"].clip(1, 1125)

print("\nAFTER")
print(f"  minimum_nights  max = {df['minimum_nights'].max():,}")
print(f"  maximum_nights  max = {df['maximum_nights'].max():,}")

# accommodates == 0 is impossible for a rentable listing.
n0 = (df["accommodates"] < 1).sum()
df["accommodates"] = df["accommodates"].clip(lower=1)
print(f"\nRepaired {n0} listings with accommodates < 1")

# bedrooms has a long tail up to 50 - plausible for a hostel, capped for stability.
df["bedrooms"] = df["bedrooms"].clip(upper=20)

### 4.5 Missing-value strategy

Each column is imputed according to its *mechanism*, per the diagnosis in §3.1. Two principles:

1. **Group-conditional medians beat global medians.** Imputing `bedrooms` with the global median
   (1.0) would tell the model that a 10-guest villa has one bedroom. Imputing from the
   `(city, accommodates)` median preserves the structural relationship.
2. **Every MNAR imputation is paired with an indicator.** The indicator is the feature that
   carries the information; the imputed value only prevents the row from being dropped. Without
   the indicator, imputation actively destroys signal.

> All group medians here are computed on the *full* dataset for exposition. In the production
> pipeline (Notebook 03) the identical logic runs **inside a fitted transformer** so that the
> statistics come from the training fold only, eliminating leakage. That parity is the whole
> point of the shared `features.py` module.

In [ ]:
REVIEW_COLS = ["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
               "review_scores_checkin", "review_scores_communication",
               "review_scores_location", "review_scores_value"]

# --- indicators FIRST, before any values are filled ---
df["has_reviews"] = df["review_scores_rating"].notna().astype("int8")
df["has_response_data"] = df["host_response_rate"].notna().astype("int8")
df["bedrooms_missing"] = df["bedrooms"].isna().astype("int8")

# --- bedrooms: group-conditional median ---
grp_med = df.groupby(["city", "accommodates"])["bedrooms"].transform("median")
df["bedrooms"] = df["bedrooms"].fillna(grp_med).fillna(1.0).clip(1, 20)

# --- review scores: city median (price levels and rating norms differ by market) ---
for c in REVIEW_COLS:
    df[c] = df[c].fillna(df.groupby("city")[c].transform("median"))
    df[c] = df[c].fillna(df[c].median())

# --- host responsiveness ---
df["host_response_rate"] = df["host_response_rate"].fillna(df["host_response_rate"].median())
df["host_acceptance_rate"] = df["host_acceptance_rate"].fillna(df["host_acceptance_rate"].median())

# --- 165 orphaned listings whose host record was deleted ---
df["host_total_listings_count"] = df["host_total_listings_count"].fillna(1.0)
for c in ["host_is_superhost", "host_has_profile_pic", "host_identity_verified"]:
    df[c] = df[c].fillna("f")

remaining = df.isna().sum()
remaining = remaining[remaining > 0]
print("Remaining nulls after imputation:")
print(remaining.to_string() if len(remaining) else "  none (host_since handled in section 5.6)")
print(f"\nIndicator prevalence:")
print(f"  has_reviews == 0       : {(df['has_reviews'] == 0).mean():.1%}")
print(f"  has_response_data == 0 : {(df['has_response_data'] == 0).mean():.1%}")
print(f"  bedrooms_missing == 1  : {(df['bedrooms_missing'] == 1).mean():.1%}")

## 5. Advanced Feature Engineering

Nine feature families are constructed. Each subsection states **what** is built, the **logical
rationale** (why this quantity should relate to price), and the **mathematical rationale** (why
this particular functional form).

### 5.1 Target transform — `log_price`

**What.** $y = \ln(\text{price\_usd})$.

**Logical rationale.** Hosts and guests reason about price multiplicatively: "20% more than the
place down the road", not "\$18 more". A model whose loss is expressed in the same terms will
make errors that are uniformly tolerable across the price range instead of concentrating them in
the expensive tail.

**Mathematical rationale.** Four properties, established empirically in §3.2:

1. **Variance stabilisation.** Price is heteroscedastic — $\mathrm{sd}(y \mid x)$ grows roughly
   proportionally with $\mathbb{E}[y \mid x]$. If $\mathrm{sd}(y) \approx c\,\mathbb{E}[y]$ then
   by the delta method $\mathrm{sd}(\ln y) \approx c$, a constant. Constant residual variance is
   precisely the assumption OLS and squared-error boosting require.
2. **Normalisation.** Skew 91.9 → 0.42, excess kurtosis 20,154 → 1.22.
3. **Additivity.** A multiplicative hedonic model becomes additive under logs, matching the
   inductive bias of both linear models and regression trees.
4. **Domain positivity.** $\exp(\hat{y}) > 0$ always.

**Cost, stated honestly.** $\mathbb{E}[\exp(\hat{y})] \neq \exp(\mathbb{E}[\hat{y}])$ by Jensen's
inequality, so naively exponentiating gives the conditional **median**, not the mean —
systematically low by a factor of $\exp(\sigma^2/2)$ under log-normal residuals, where $\sigma$ is
the **residual** standard deviation. Notebook 02 measures $\sigma \approx 0.434$ for the best
model, giving a 9.9% understatement of the conditional mean. This is quantified and corrected in
Notebook 02, §9.2. Reporting the median is often the *preferred* behaviour for a price
recommender, but it must be a deliberate choice rather than an accident.

In [ ]:
df[TARGET_MODEL] = np.log(df[TARGET_USD])
sigma_hint = df[TARGET_MODEL].std()
print(f"log_price: mean={df[TARGET_MODEL].mean():.3f}  sd={sigma_hint:.3f}  "
      f"skew={df[TARGET_MODEL].skew():.3f}")
print(f"Smearing factor exp(sd^2/2) if residual sd were {sigma_hint:.3f}: "
      f"{np.exp(sigma_hint ** 2 / 2):.3f}")
print("(The real correction uses the RESIDUAL sd, computed in Notebook 02.)")

### 5.2 Geospatial features — haversine distance to city centre

**What.** Great-circle distance in km from each listing to its own city's centre, plus a
log-compressed version.

**Logical rationale.** Access to the centre is the classic driver of urban land rent (Alonso–
Muth–Mills bid-rent theory). Raw latitude/longitude are *not* interchangeable with this: 48.85°N
means "central" in Paris and "nowhere" in Sydney, so a tree must waste splits reconstructing a
per-city origin. Distance is city-relative by construction, so it transfers across cities and
lets one split serve all ten.

**Mathematical rationale.** Degrees of longitude are not a constant distance — one degree is
73 km in Paris and 108 km in Bangkok. Euclidean distance on raw coordinates would therefore be
systematically distorted between cities. The haversine formula gives true metric distance on a
sphere:

$$a = \sin^2\!\Big(\frac{\Delta\varphi}{2}\Big) + \cos\varphi_1\cos\varphi_2\sin^2\!\Big(\frac{\Delta\lambda}{2}\Big), \qquad
d = 2R\arcsin\!\big(\sqrt{a}\big)$$

with $R = 6{,}371$ km. It is numerically stable for the small distances involved here (unlike the
spherical law of cosines, which loses precision below ~1 km).

We also add $\ln(1+d)$. Bid-rent decay is approximately exponential in distance, so the log makes
the relationship closer to linear — this matters for the Ridge baseline, which cannot bend a
straight line. Raw `latitude`/`longitude` are **retained as well**, because §3.4 showed the price
surface is non-monotonic (waterfront premia at 20 km+) and a purely radial feature cannot express
that. The tree models use coordinates to carve out those pockets.

In [ ]:
df["dist_centre_km"] = haversine_km(
    df["latitude"].values, df["longitude"].values,
    df["city"].map(lambda c: CITY_CENTRE[c][0]).values,
    df["city"].map(lambda c: CITY_CENTRE[c][1]).values)
df["log_dist_centre"] = np.log1p(df["dist_centre_km"])

print(df.groupby("city")["dist_centre_km"].describe(percentiles=[0.5, 0.9])
      [["mean", "50%", "90%", "max"]].round(2).to_string())
print(f"\nCorrelation with log_price: raw {df['dist_centre_km'].corr(df[TARGET_MODEL]):+.3f}  "
      f"| log {df['log_dist_centre'].corr(df[TARGET_MODEL]):+.3f}")

### 5.3 Amenity decomposition — count, targeted flags, and a luxury score

**What.** The `amenities` column is a JSON array with **3,446 distinct values** across the
corpus. It is decomposed into (a) a total count, (b) 23 binary flags for amenities with a clear
economic story, and (c) an aggregate luxury score.

**Logical rationale.** The raw column is unusable as-is: one-hot encoding 3,446 categories would
produce a matrix wider than the dataset is deep, with a power-law frequency distribution in which
most columns are near-constant. Three complementary summaries capture the useful variation:

- **`n_amenities`** — a proxy for how completely the host has furnished and *documented* the
  listing. It measures effort and professionalism as much as physical inventory.
- **Targeted flags** — a small, hand-picked set (pool, dishwasher, air conditioning, elevator,
  bathtub, gym, hot tub, free parking, …) chosen because each has a defensible price mechanism.
- **`luxury_score`** — the count of nine capital-intensive amenities.

**Mathematical rationale for the aggregate.** Individual luxury flags are sparse (`hot_tub` 4.7%,
`bathtub` 6.8%) so each is a high-variance, low-information column that a tree will rarely split
on. Summing $k$ correlated Bernoulli indicators produces a variable with higher variance and a
better signal-to-noise ratio: for indicators with common variance $\sigma^2$ and mean pairwise
correlation $\rho$,

$$\mathrm{Var}\Big(\textstyle\sum_{j=1}^{k} x_j\Big) = k\sigma^2\big(1 + (k-1)\rho\big)$$

which grows super-linearly in $k$ when $\rho > 0$. The aggregate is a single well-populated
ordinal feature usable by every model class, and it is more robust to the city-confounding
demonstrated in §3.6 than any single flag.

**Caveat carried forward.** Per §3.6, individual amenity coefficients are contaminated by city
and must not be interpreted causally ("install heating, earn 69% more" is false).

In [ ]:
AMENITY_FLAGS = {
    "pool": "Pool", "aircon": "Air conditioning", "dishwasher": "Dishwasher",
    "elevator": "Elevator", "free_parking": "Free parking on premises",
    "washer": "Washer", "dryer": "Dryer", "gym": "Gym", "hot_tub": "Hot tub",
    "workspace": "Dedicated workspace", "self_checkin": "Self check-in",
    "bathtub": "Bathtub", "bbq": "BBQ grill", "breakfast": "Breakfast",
    "tv": "TV", "wifi": "Wifi", "kitchen": "Kitchen", "heating": "Heating",
    "patio": "Patio or balcony", "crib": "Crib", "longterm": "Long term stays allowed",
    "smoke_alarm": "Smoke alarm", "pets": "Pets allowed",
}
LUXURY_SET = ["pool", "hot_tub", "gym", "dishwasher", "bathtub",
              "bbq", "elevator", "free_parking", "aircon"]


def parse_amenities(series: pd.Series) -> pd.DataFrame:
    """Single pass over the JSON column -> count + one column per targeted flag."""
    raw = series.fillna("[]").values
    n = len(raw)
    counts = np.zeros(n, dtype="int16")
    flags = {k: np.zeros(n, dtype="int8") for k in AMENITY_FLAGS}
    for i, v in enumerate(raw):
        try:
            items = set(json.loads(v))
        except (ValueError, TypeError):
            items = set()
        counts[i] = len(items)
        for key, label in AMENITY_FLAGS.items():
            if label in items:
                flags[key][i] = 1
    out = pd.DataFrame({"n_amenities": counts}, index=series.index)
    for key, arr in flags.items():
        out[f"am_{key}"] = arr
    return out


am = parse_amenities(df["amenities"])
df = pd.concat([df.drop(columns=["amenities"]), am], axis=1)
df["luxury_score"] = df[[f"am_{k}" for k in LUXURY_SET]].sum(axis=1).astype("int8")

print(f"Amenity features created: {am.shape[1] + 1}")
print(f"n_amenities  : mean {df['n_amenities'].mean():.1f}, "
      f"median {df['n_amenities'].median():.0f}, max {df['n_amenities'].max()}")
print(f"luxury_score : mean {df['luxury_score'].mean():.2f}, "
      f"corr with log_price {df['luxury_score'].corr(df[TARGET_MODEL]):+.3f}")
print("\nWithin-city correlation of luxury_score with log_price:")
print(df.groupby("city").apply(
    lambda g: g["luxury_score"].corr(g[TARGET_MODEL])).round(3).to_string())

### 5.4 Capacity density — `persons_per_bedroom`

**What.** $\text{persons\_per\_bedroom} = \dfrac{\text{accommodates}}{\text{bedrooms}}$, plus
`is_studio` and `is_large_group` indicators.

**Logical rationale.** Two listings that both sleep 6 are entirely different products: three
bedrooms at 2 people each is a family apartment; one bedroom with sofa beds at 6 people is a
budget-share. They occupy different price tiers and attract different guests. Neither
`accommodates` nor `bedrooms` alone distinguishes them; the *ratio* does.

**Mathematical rationale.** `accommodates` and `bedrooms` correlate at ~0.75, so a linear model
sees them as near-collinear and splits the coefficient arbitrarily between them. The ratio is a
**non-linear interaction that no additive model can construct on its own** — a linear model can
represent $\beta_1 a + \beta_2 b$ but never $a/b$. Supplying it explicitly hands the linear
baseline a piece of structure it would otherwise be architecturally incapable of learning, which
is exactly the point of feature engineering.

For tree models the ratio is a labour saver rather than an impossibility: a tree *can*
approximate $a/b$, but only by stacking many axis-aligned splits, spending depth budget that is
better used elsewhere.

Ratios are also **scale-free**, so a value of 2.0 means the same thing in a studio and in a
mansion — an economical use of a single split point.

In [ ]:
df["persons_per_bedroom"] = (df["accommodates"] / df["bedrooms"]).clip(0.25, 8)
df["is_studio"] = ((df["bedrooms"] == 1) & (df["accommodates"] <= 2)).astype("int8")
df["is_large_group"] = (df["accommodates"] >= 7).astype("int8")

print("Median log_price by (bedrooms, persons_per_bedroom band) for 6-guest listings:")
six = df[df["accommodates"] == 6].copy()
six["ppb_band"] = pd.cut(six["persons_per_bedroom"], [0, 1.5, 2.5, 10],
                         labels=["<=1.5 spacious", "1.5-2.5 standard", ">2.5 dense"])
print(six.groupby("ppb_band", observed=True).agg(
    n=(TARGET_USD, "size"), median_usd=(TARGET_USD, "median")).round(1).to_string())
print("\nSame guest capacity, materially different price -> the ratio carries")
print("information that `accommodates` alone cannot express.")

### 5.5 Host professionalisation

**What.** `log_host_listings`, `is_professional_host`, `host_tenure_days`.

**Logical rationale.** A host with 40 listings is a property-management business running yield
management; a host with one listing is renting a spare room. They price differently — the
professional prices to a model, the amateur prices to a neighbour's guess. §3.5 showed this
effect is real (+19% within city) but *invisible* in pooled data.

**Mathematical rationale for the log.** `host_total_listings_count` is extremely right-skewed
(median 1, maximum in the hundreds). Under a raw coding, the distance from 1 to 2 listings equals
the distance from 300 to 301 — but the first is a qualitative change in business model and the
second is a rounding error. $\ln(1+x)$ imposes the correct diminishing-returns geometry: equal
*ratios* map to equal distances. $\ln(1+x)$ rather than $\ln x$ so that a single-listing host
maps to 0 rather than being undefined.

`is_professional_host` (>5 listings) is added because the effect has a **threshold** character —
crossing from hobbyist into business — and a binary lets the model place a step change without
spending a split on locating it.

In [ ]:
df["host_since"] = pd.to_datetime(df["host_since"], errors="coerce")
df["host_tenure_days"] = (SNAPSHOT_DATE - df["host_since"]).dt.days
df["host_tenure_days"] = df["host_tenure_days"].fillna(df["host_tenure_days"].median()).clip(lower=0)

df["log_host_listings"] = np.log1p(df["host_total_listings_count"])
df["is_professional_host"] = (df["host_total_listings_count"] > 5).astype("int8")

lb = pd.cut(df["host_total_listings_count"], [0, 1, 2, 5, 20, 1e6],
            labels=["1", "2", "3-5", "6-20", "21+"])
print("Median USD price by host portfolio size, WITHIN Paris (removes city confounding):")
par = df[df["city"] == "Paris"]
print(par.groupby(lb[par.index], observed=True).agg(
    n=(TARGET_USD, "size"), median_usd=(TARGET_USD, "median")).round(1).to_string())
print(f"\nis_professional_host share: {df['is_professional_host'].mean():.1%}")
print(f"host_tenure_days: median {df['host_tenure_days'].median():,.0f} "
      f"({df['host_tenure_days'].median() / 365:.1f} years)")

### 5.6 Review composites and the value gap

**What.** `review_composite` (mean of six sub-scores), `value_gap`, and
`location_premium_signal`.

**Logical rationale.** The seven review columns are mutually correlated at 0.6–0.8 and share a
single dominant factor ("was it good?"). Feeding all seven gives the model seven noisy views of
one construct. The informative part is not the *level* but the *deviations between* sub-scores —
a listing rated 9.8 on location and 8.5 on cleanliness is a specific product profile.

**Mathematical rationale.** Decompose each sub-score into a common factor plus a residual:

$$s_j = \mu + \delta_j, \qquad \mu = \tfrac{1}{6}\textstyle\sum_j s_j$$

Then $\mu$ (`review_composite`) captures the shared quality factor with variance reduced by
averaging — for six sub-scores of equal variance $\sigma^2$ and mean correlation $\rho$,

$$\mathrm{Var}(\mu) = \frac{\sigma^2}{6}\big(1 + 5\rho\big)$$

which at $\rho \approx 0.7$ retains ~75% of the original variance while cancelling a substantial
share of the idiosyncratic rating noise. The $\delta_j$ terms then carry the orthogonal,
profile-specific information. We keep two of them:

- **`value_gap`** $= s_{\text{value}} - s_{\text{overall}}/10$ — the price-perception signal
  established in §3.7 (monotone from +0.095 to −0.126 across price quintiles).
- **`location_premium_signal`** $= s_{\text{location}} - \mu$ — how much better the location is
  than the listing overall, a guest-validated location quality measure that is independent of
  the coordinate-based features.

**Leakage note.** Both differences are computed from reviews written *after* the price was set,
so they are downstream of the target. They are valid for an estimation model but must be excluded
from counterfactual "what should I charge?" use. Flagged in the model card.

In [ ]:
SUB_SCORES = ["review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin",
              "review_scores_communication", "review_scores_location", "review_scores_value"]

df["review_composite"] = df[SUB_SCORES].mean(axis=1)
df["value_gap"] = df["review_scores_value"] - df["review_scores_rating"] / 10.0
df["location_premium_signal"] = df["review_scores_location"] - df["review_composite"]

RESP_ORDER = {"within an hour": 1, "within a few hours": 2,
              "within a day": 3, "a few days or more": 4}
df["response_speed"] = df["host_response_time"].map(RESP_ORDER).fillna(5).astype("int8")

print("Variance retained by the composite:")
print(f"  mean sub-score variance : {df[SUB_SCORES].var().mean():.4f}")
print(f"  composite variance      : {df['review_composite'].var():.4f}")
print(f"  mean pairwise corr      : "
      f"{df[SUB_SCORES].corr().values[np.triu_indices(6, 1)].mean():.3f}")
print(f"\nvalue_gap corr with log_price               : {df['value_gap'].corr(df[TARGET_MODEL]):+.3f}")
print(f"location_premium_signal corr with log_price : "
      f"{df['location_premium_signal'].corr(df[TARGET_MODEL]):+.3f}")
print(f"review_composite corr with log_price        : "
      f"{df['review_composite'].corr(df[TARGET_MODEL]):+.3f}")
print("\nThe composite (the shared 'quality' factor) is nearly uncorrelated with")
print("price, while the DIFFERENCES carry signal. Averaging alone would have")
print("thrown away the useful part.")

### 5.7 Stay-policy regime features

**What.** `log_min_nights`, `is_monthly_only`, `booking_window`.

**Logical rationale.** §3.3 established that listings with `minimum_nights >= 30` are competing
in the monthly-rental market, where the nightly rate is a *derived* quantity (monthly rent ÷ 30)
rather than a directly-set one. Pooling the two regimes without a marker forces the model to fit
a blend of two different price-formation processes.

There is a second, more interesting reason, and it is another instance of the confounding pattern
from §3.5. The long-stay regime is not evenly distributed: **66.7% of New York listings have a
30-night minimum, against 1.6–3.7% in most other cities.** That is not host preference, it is
regulation — New York's Multiple Dwelling Law effectively prohibits short-term rental of entire
units, so hosts set a 30-night floor to stay compliant. Because New York is also one of the most
expensive markets, the *pooled* comparison says long-stay listings are more expensive
(\$88.50 vs \$70.80 median). *Within* almost every city the opposite is true — Rome \$77.90 → \$53.10,
Cape Town \$73.70 → \$50.40, Mexico City \$33.60 → \$25.40. The pooled figure is a composition effect
produced by a single city's regulatory regime.

This is precisely why `is_monthly_only` must be paired with `city` in the model rather than used
as a standalone signal, and it is a concrete reminder that **legal and regulatory context is a
latent variable in this dataset** — one the model can only capture indirectly, through the city
term.

**Mathematical rationale.** `minimum_nights` clusters hard at 1, 2, 3, 7, 30 and 90 — it is
effectively ordinal with a power-law tail, not a smooth continuum. $\ln(1+x)$ compresses the tail
so that the meaningful gaps (1→2 nights, 2→3 nights) occupy comparable distance to the
irrelevant ones (300→365). The `is_monthly_only` binary then supplies the **regime intercept**:
a single split that lets the model estimate a separate baseline for the long-stay market instead
of averaging across a genuine discontinuity.

In [ ]:
df["log_min_nights"] = np.log1p(df["minimum_nights"])
df["is_monthly_only"] = (df["minimum_nights"] >= 30).astype("int8")
df["booking_window"] = (df["maximum_nights"] - df["minimum_nights"]).clip(lower=0)

print(df.groupby("is_monthly_only").agg(
    n=(TARGET_USD, "size"),
    median_usd=(TARGET_USD, "median"),
    mean_accommodates=("accommodates", "mean"),
    pct_entire_place=("room_type", lambda s: (s == "Entire place").mean())).round(2).to_string())
print("\nPooled: long-stay listings look MORE expensive ($88.5 vs $70.8).")
print("Now split by city — the pooled result reverses:")
print(df.groupby(["city", "is_monthly_only"])[TARGET_USD].median().unstack().round(1).to_string())
print("\nShare of listings that are long-stay-only, by city:")
print(df.groupby("city")["is_monthly_only"].mean().sort_values(ascending=False).round(3).to_string())
print("\nNew York at 66.7% is a regulatory artefact (Multiple Dwelling Law),")
print("not a host preference. The pooled comparison is a composition effect.")

### 5.8 Categorical consolidation

**What.** `property_type` (144 levels) → `property_grouped` (top 15 + `Other`).

**Logical rationale.** The frequency distribution is a severe power law: `Entire apartment`
covers 50% of rows while the tail contains levels appearing once ("Entire castle", "Private room
in windmill"). A level with 3 observations cannot support a reliable estimate — whatever the
model learns for it is noise that will not generalise.

**Mathematical rationale.** For a category with $n_g$ observations, the standard error of its
group mean is $\sigma/\sqrt{n_g}$. At $n_g = 3$ and $\sigma = 0.867$ (the observed log-price
spread) that is $\pm 0.50$ in log units — an interval spanning a factor of 2.7 in price, wider
than the between-category differences we are trying to estimate. In this dataset **81 of the 141
property types have fewer than 30 rows and together cover only 0.21% of listings**: 81 columns of
almost pure noise. Such categories add variance without
adding signal. Consolidating them into `Other` pools their observations so the group mean is
estimated with usable precision, and it bounds the one-hot matrix width, which matters directly
for the Ridge baseline's condition number.

In [ ]:
TOP_N_PROPERTY = 15
top_types = df["property_type"].value_counts().head(TOP_N_PROPERTY).index
df["property_grouped"] = np.where(df["property_type"].isin(top_types),
                                  df["property_type"], "Other")

vc = df["property_type"].value_counts()
print(f"property_type levels     : {df['property_type'].nunique()}")
print(f"  levels with < 30 rows  : {(vc < 30).sum()}  "
      f"covering {vc[vc < 30].sum():,} listings ({vc[vc < 30].sum() / len(df):.2%})")
print(f"property_grouped levels  : {df['property_grouped'].nunique()}")
print(f"  'Other' share          : {(df['property_grouped'] == 'Other').mean():.2%}")

for c in ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "instant_bookable"]:
    df[c] = (df[c] == "t").astype("int8")
print("\nBinary t/f columns encoded to 0/1.")

## 6. Train / test split — stratified by city, *before* target encoding

The split is deliberately placed **before** the neighbourhood price index (§7) is constructed,
because that feature uses the target and must never see test-set prices.

**Why stratify by city.** §3.5 showed that city is the dominant confounder. If one split ended up
with 3% more Paris listings than the other, the difference in mean price would be attributed to
the model rather than to sampling. Stratification pins the city composition and, just as
importantly, guarantees every city is represented in the test set so that **per-city error can be
reported** — a fairness requirement, not just a statistical nicety.

**A limitation we accept and record.** A host can own many listings, so a purely random split
places some of a host's listings in train and others in test. Those are not independent
observations, which optimistically biases the test score. A `GroupShuffleSplit` on `host_id`
would remove this, at the cost of breaking city stratification and shrinking the effective test
set. We keep the stratified split for comparability with the reference notebooks, quantify the
exposure below (43.5% of test rows share a host with a training row), and flag it as a known
limitation in the final report.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = [
    # capacity
    "accommodates", "bedrooms", "persons_per_bedroom", "is_studio", "is_large_group",
    # geography
    "latitude", "longitude", "dist_centre_km", "log_dist_centre",
    # amenities
    "n_amenities", "luxury_score", *[f"am_{k}" for k in AMENITY_FLAGS],
    # host
    "host_tenure_days", "log_host_listings", "is_professional_host",
    "host_is_superhost", "host_identity_verified", "host_has_profile_pic",
    "host_response_rate", "host_acceptance_rate", "response_speed",
    # reviews
    "review_scores_rating", "review_composite", "value_gap", "location_premium_signal",
    "review_scores_cleanliness", "review_scores_location",
    # policy
    "log_min_nights", "is_monthly_only", "booking_window", "instant_bookable",
    # missingness indicators
    "has_reviews", "has_response_data", "bedrooms_missing",
    # categoricals
    "city", "room_type", "property_grouped",
    # held for target encoding (dropped after section 7)
    "neighbourhood",
]

X = df[FEATURE_COLS].copy()
y = df[TARGET_MODEL].copy()
meta = df[["city", TARGET_USD]].copy()
meta["host_id"] = HOST_IDS.reindex(df.index).values

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y, meta, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["city"])

print(f"Train : {X_train.shape[0]:,} rows x {X_train.shape[1]} columns")
print(f"Test  : {X_test.shape[0]:,} rows")
print(f"\nCity composition (train vs test %):")
comp = pd.DataFrame({
    "train_%": X_train["city"].value_counts(normalize=True) * 100,
    "test_%": X_test["city"].value_counts(normalize=True) * 100})
comp["diff"] = (comp["train_%"] - comp["test_%"]).abs()
print(comp.round(3).to_string())
print(f"\nMax composition drift: {comp['diff'].max():.3f} pp — stratification confirmed.")

shared = set(meta_train["host_id"]) & set(meta_test["host_id"])
exposed = meta_test["host_id"].isin(shared).mean()
print(f"\n[LIMITATION] Hosts appearing in both splits: {len(shared):,}")
print(f"             Test rows from such hosts       : {exposed:.1%}")
print("             -> test score is mildly optimistic; recorded in the model card.")

## 7. The neighbourhood price index — smoothed target encoding

This is the highest-value engineered feature in the project (η² = 0.32 on its own, §2.3) and also
the most dangerous, so it gets its own section.

**What.** Replace the 660-level `neighbourhood` string with a single numeric column: the mean
log-price of that neighbourhood, shrunk toward the global mean in proportion to how little data
supports it.

**Why not one-hot.** 660 columns for 221k training rows, with a power-law frequency distribution
in which the smallest neighbourhood has 1 observation. The rare columns are pure variance.

**Why not a plain group mean.** A neighbourhood with 2 listings would get a mean estimated from
2 observations and the model would trust it exactly as much as one estimated from 5,000.

**The mathematics — empirical-Bayes shrinkage.** Treat each neighbourhood mean as drawn from a
population of neighbourhood means. The posterior estimate is a precision-weighted blend of the
group mean and the prior:

$$\hat{\mu}_g = \frac{n_g \bar{y}_g + k\,\bar{y}}{n_g + k}
             = w_g\,\bar{y}_g + (1 - w_g)\,\bar{y}, \qquad w_g = \frac{n_g}{n_g + k}$$

where $n_g$ is the neighbourhood's row count, $\bar{y}_g$ its mean, $\bar{y}$ the global mean and
$k$ the smoothing strength. $k$ has a direct interpretation: it is the number of pseudo-
observations of the prior added to every group, and it is the value of $n_g$ at which the
estimate sits exactly halfway between group mean and prior. We set $k = 20$: a neighbourhood with
20 listings gets 50% weight, one with 500 gets 96%, one with 2 gets 9%. Under a Gaussian
hierarchical model $k = \sigma^2_{\text{within}} / \sigma^2_{\text{between}}$, so this is a
principled quantity, not an arbitrary constant.

**The leakage control, which matters more than the formula.** Target encoding uses $y$ to build
$X$. Done naively, each row's own target contributes to its own feature, and a tree will exploit
that to memorise the training set. Two defences are applied together:

1. **Fit on the training split only.** The test set is encoded using the *training* mapping. It
   never contributes to any statistic.
2. **Out-of-fold encoding within the training set.** For the training rows themselves, each row's
   encoded value is computed from the *other* $K-1$ folds. A row therefore never sees its own
   target, and the training-time distribution of the feature matches what the model will see at
   inference. Without this, the model's training error collapses and hyperparameter selection is
   driven by an artefact.

Unseen neighbourhoods at inference time fall back to the global prior.

In [ ]:
from sklearn.model_selection import KFold


class SmoothedTargetEncoder:
    """Empirical-Bayes target encoder with out-of-fold training values.

    Parameters
    ----------
    k : float
        Smoothing strength. The number of prior pseudo-observations added to
        every group; also the group size at which the estimate is halfway
        between the group mean and the global prior.
    n_folds : int
        Folds used to generate leakage-free encodings for the training rows.
    """

    def __init__(self, k: float = 20.0, n_folds: int = 5, random_state: int = 42):
        self.k = k
        self.n_folds = n_folds
        self.random_state = random_state

    def _fit_map(self, keys: pd.Series, target: pd.Series) -> dict:
        stats = pd.DataFrame({"k": keys.values, "y": target.values}).groupby("k")["y"]
        agg = stats.agg(["mean", "count"])
        blended = (agg["mean"] * agg["count"] + self.prior_ * self.k) / (agg["count"] + self.k)
        return blended.to_dict()

    def fit(self, keys: pd.Series, target: pd.Series):
        self.prior_ = float(target.mean())
        self.mapping_ = self._fit_map(keys, target)
        return self

    def transform(self, keys: pd.Series) -> pd.Series:
        return keys.map(self.mapping_).astype(float).fillna(self.prior_)

    def fit_transform_oof(self, keys: pd.Series, target: pd.Series) -> pd.Series:
        """Fit the full mapping, but return OUT-OF-FOLD values for these rows."""
        self.fit(keys, target)
        oof = pd.Series(np.nan, index=keys.index, dtype=float)
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        for tr_idx, va_idx in kf.split(keys):
            fold_prior = float(target.iloc[tr_idx].mean())
            fold = SmoothedTargetEncoder(self.k, self.n_folds, self.random_state)
            fold.prior_ = fold_prior
            fold.mapping_ = fold._fit_map(keys.iloc[tr_idx], target.iloc[tr_idx])
            oof.iloc[va_idx] = fold.transform(keys.iloc[va_idx]).values
        return oof.fillna(self.prior_)


nbhd_encoder = SmoothedTargetEncoder(k=20.0, n_folds=5, random_state=RANDOM_STATE)

X_train["nbhd_price_index"] = nbhd_encoder.fit_transform_oof(
    X_train["neighbourhood"], y_train)                      # out-of-fold
X_test["nbhd_price_index"] = nbhd_encoder.transform(
    X_test["neighbourhood"])                                # train-fitted mapping only

unseen = (~X_test["neighbourhood"].isin(nbhd_encoder.mapping_)).sum()
print(f"Encoder fitted on {len(nbhd_encoder.mapping_)} training neighbourhoods")
print(f"Global prior (mean log price) : {nbhd_encoder.prior_:.4f}")
print(f"Test rows with unseen neighbourhood -> prior: {unseen}")

sizes = X_train["neighbourhood"].value_counts()
demo = pd.DataFrame({
    "n_train": sizes,
    "raw_mean": y_train.groupby(X_train["neighbourhood"]).mean(),
    "encoded": pd.Series(nbhd_encoder.mapping_)})
demo["weight_w"] = demo["n_train"] / (demo["n_train"] + nbhd_encoder.k)
print("\nShrinkage in action — largest and smallest neighbourhoods:")
print(pd.concat([demo.nlargest(3, "n_train"), demo.nsmallest(3, "n_train")]).round(3).to_string())

X_train = X_train.drop(columns=["neighbourhood"])
X_test = X_test.drop(columns=["neighbourhood"])
print(f"\nFinal matrix: {X_train.shape[1]} features")

**Observed result**

```text
Encoder fitted on 653 training neighbourhoods
Global prior (mean log price) : 4.2749
Test rows with unseen neighbourhood -> prior: 7

Shrinkage in action:
neighbourhood       n_train   raw_mean   encoded   weight_w
I Centro Storico     11,821      4.566     4.566      0.998   <- essentially unshrunk
Sydney                6,402      4.542     4.541      0.997
Copacabana            6,150      4.008     4.009      0.997
Acari                     1      3.861     4.255      0.048   <- pulled 95% back to the prior
Castle Hill               1      4.263     4.274      0.048
```

`Acari` is the case the shrinkage exists for. Its single training listing is priced well below
the global average; without shrinkage the model would be handed "this neighbourhood is cheap" as
a confident fact derived from one observation. With $k = 20$ it receives 4.8% weight and the
encoded value sits essentially at the prior — the model is told, correctly, that we know nothing
about Acari. Seven test rows fall in neighbourhoods absent from the training split entirely and
fall back to the prior by the same logic.

The two extremes show the mechanism working exactly as intended: well-observed neighbourhoods
keep their own mean; thinly-observed ones are pulled almost all the way back to the global prior,
so the model is not handed a confident-looking number derived from two listings.

**A caveat established in Notebook 02.** Permutation importance will rank this feature first by a
wide margin — yet an *ablation* (retraining without it) barely changes test R² (0.7454 → 0.7459).
The information is redundant with `latitude`, `longitude` and `city`, which the tree models can
combine to reconstruct the same spatial price surface. This is a textbook illustration of why
permutation importance is unreliable under correlated features, and it is the reason the final
report distinguishes *attribution* from *necessity*.

## 8. Save processed data to S3 (Pipeline stage 1 — Data Storage)

In [ ]:
def save_csv_to_s3(obj, bucket: str, key: str) -> str:
    buf = io.StringIO()
    if isinstance(obj, pd.Series):
        obj.to_csv(buf, index=False, header=True)
    else:
        obj.to_csv(buf, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    return f"s3://{bucket}/{key}"


paths = {
    "train_features": save_csv_to_s3(X_train, BUCKET, f"{PROCESSED_PREFIX}/train_features.csv"),
    "train_labels": save_csv_to_s3(y_train.rename(TARGET_MODEL), BUCKET,
                                   f"{PROCESSED_PREFIX}/train_labels.csv"),
    "test_features": save_csv_to_s3(X_test, BUCKET, f"{PROCESSED_PREFIX}/test_features.csv"),
    "test_labels": save_csv_to_s3(y_test.rename(TARGET_MODEL), BUCKET,
                                  f"{PROCESSED_PREFIX}/test_labels.csv"),
    "test_meta": save_csv_to_s3(meta_test, BUCKET, f"{PROCESSED_PREFIX}/test_meta.csv"),
}

# The fitted encoder must travel with the data — Notebook 03 rebuilds it inside the pipeline.
s3.put_object(Bucket=BUCKET, Key=f"{PROCESSED_PREFIX}/nbhd_encoder.json",
              Body=json.dumps({"k": nbhd_encoder.k, "prior": nbhd_encoder.prior_,
                               "mapping": nbhd_encoder.mapping_}, indent=2))

feature_schema = {
    "target_model": TARGET_MODEL, "target_usd": TARGET_USD,
    "fx_version": FX_VERSION, "snapshot_date": str(SNAPSHOT_DATE.date()),
    "n_features": int(X_train.shape[1]),
    "feature_order": list(X_train.columns),
    "categorical": ["city", "room_type", "property_grouped"],
    "binary": [c for c in X_train.columns if X_train[c].dropna().isin([0, 1]).all()],
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "trim_quantiles": [0.005, 0.995],
    "random_state": RANDOM_STATE,
}
s3.put_object(Bucket=BUCKET, Key=f"{PROCESSED_PREFIX}/feature_schema.json",
              Body=json.dumps(feature_schema, indent=2))

for name, uri in paths.items():
    print(f"{name:<16}: {uri}")
print(f"{'nbhd_encoder':<16}: s3://{BUCKET}/{PROCESSED_PREFIX}/nbhd_encoder.json")
print(f"{'feature_schema':<16}: s3://{BUCKET}/{PROCESSED_PREFIX}/feature_schema.json")
print(f"{'fx_rates':<16}: s3://{BUCKET}/{PROCESSED_PREFIX}/fx_rates.json")
print(f"\nProcessed prefix: s3://{BUCKET}/{PROCESSED_PREFIX}")
print("Next: open Notebook 02 for baseline experiments and MLflow tracking.")

## 9. Governance flags raised during data preparation

These are recorded here, at the point of discovery, and carried into the AI Governance section of
the final report. Framing follows Singapore's **IMDA AI Verify** / **ISAGO** testing dimensions.

| # | Flag | Dimension | Evidence | Mitigation |
|---|---|---|---|---|
| G1 | **Target is asking price, not transacted price** | Robustness / validity | No booking or occupancy data in the source | Model card states the model predicts *listed rates*, not market-clearing prices. Any "optimal price" claim is out of scope. |
| G2 | **FX rates are an analyst assumption** | Transparency / reproducibility | `FX_RATES` is hand-supplied; no rate source in the data | Rates versioned (`fx_2021Q1_v1`), persisted to S3, logged as an MLflow parameter, and surfaced in the model card. |
| G3 | **Cold-start cohort is 32.7% of the data** | Fairness / inclusiveness | Listings with no reviews have 273 fewer days of median host tenure | Retained with a `has_reviews` indicator; per-segment error reported in Notebook 02 §9.7. |
| G4 | **Feedback-loop risk** | Robustness | If the model recommends prices and hosts adopt them, next year's training data contains the model's own output | Documented; requires a champion/challenger holdout and drift monitoring in production (Notebook 03 §9). |
| G5 | **Neighbourhood encoding can proxy for socio-economic status** | Fairness | `nbhd_price_index` is the strongest single feature (η² 0.32) | Neighbourhood is legitimate in property valuation, but per-city and per-price-tier error parity is tested explicitly. Recorded as a monitored feature. |
| G6 | **Interpretability limits of amenity effects** | Explainability | §3.6: "Heating +69%" is a city proxy, not a causal effect | Model card prohibits presenting single-feature effects as ROI advice to hosts. |
| G7 | **Host-level grouping violated by the split** | Validity | 14,610 hosts appear in both splits; 43.5% of test rows share a host with a training row | Quantified in §6; test metrics flagged as mildly optimistic. |
| G8 | **Geographic coordinates are personal-adjacent data** | Privacy | Latitude/longitude locate a dwelling to metres | Coordinates are already public on the platform; `host_id`/`listing_id` are excluded from the feature matrix and retained only as an audit trail. |

---
## Checklist before Notebook 02

- [x] Raw snapshot version-pinned in S3
- [x] Data suitability formally assessed (S1–S4) — conditional pass on FX normalisation
- [x] Currency normalisation applied and the assumption persisted as a versioned artefact
- [x] Outlier policy applied per city, with a sensitivity check and a stated applicability boundary
- [x] Missing values handled by mechanism (MNAR indicators added, not just imputation)
- [x] Sentinel values repaired (`minimum_nights`, `maximum_nights`, `accommodates`)
- [x] Nine engineered feature families, each with logical and mathematical justification
- [x] Split stratified by city, performed **before** target encoding
- [x] Neighbourhood index encoded out-of-fold on train, train-mapping-only on test
- [x] Processed matrices, encoder, feature schema and FX table written to S3
- [x] Eight governance flags recorded with evidence and mitigations

**Headline EDA findings to carry forward**

1. `price` is recorded in ten different currencies — the finding that makes or breaks the project.
2. `log(price)` is the correct target: skew 91.9 → 0.42, and log-scale error *is* relative error.
3. Location dominates (η² = 0.32 for neighbourhood alone, above `city` at 0.22); capacity is second.
4. Simpson's paradox appears three separate times — host professionalisation, amenity lift, and
   the long-stay regime. `city` must be in the model and all screening must be done within city.
5. The superhost premium is not real at population level; its sign flips between markets.
6. Distance decay is non-monotonic (peak at 1–2 km, second rise beyond 20 km) — a strong prior in
   favour of tree ensembles over linear models.
7. Guest-perceived value declines monotonically with price even as overall ratings rise, giving a
   usable (but non-counterfactual) over-pricing signal.